# Śrīmad Bhāgavata Purāṇa: Corpus Builder

Builds a single, verse-aligned JSON record of the **entire Bhāgavata Purāṇa**:

* **Original text** in Devanāgarī (plus IAST and ITRANS), sloka by sloka, from
  [sanskritdocuments.org](https://sanskritdocuments.org/doc_purana/bhagpur.itx).
* **Two independent English translations** for every sloka, scraped and aligned
  to the canonical `skandha.chapter.verse` reference.

### Structure preserved

The canonical hierarchy is **Skandha (12) → Adhyāya (335) → Śloka (14,088) → Pāda**, and the
JSON uses exactly those names: `skandha`, `adhyaya`, `sloka`, `pada`. Every verse is addressed
by a stable ID of the form `10.14.8` (skandha 10, adhyāya 14, śloka 8), and its metrical
quarter-lines are retained in `text.padas`.

### Vachanas are separated, not deleted

Speaker attributions and prose lead-ins such as `śrī-śuka uvāca`, `rājovāca`,
`ṛṣaya ūcuḥ` and `tatrāyaṃ ślokaḥ` are **not part of the sloka**. The source encodes them as pāda `0`,
so they are extracted into a separate `vachana` field on the record that they introduce,
typed as `speaker` or `lead_in`. The `text` field therefore contains verse text only.

There are **no** `atha prathamo'dhyāyaḥ` / `iti ... skandhaḥ` chapter headings or colophons
to strip: the sanskritdocuments ITRANS edition carries no running headers at all, encoding
the structure purely in a numeric line prefix (verified in §5).

### Sources

| Role | Source | Edition / translator | Notes |
|---|---|---|---|
| Sanskrit text | `sanskritdocuments.org/doc_purana/bhagpur.itx` | ITRANS e-text (vol. contributed by Ulrich) | Numerically indexed, machine-parseable |
| Translation A | [wisdomlib.org](https://www.wisdomlib.org/hinduism/book/the-bhagavata-purana) | **G. V. Tagare**, *The Bhāgavata Purāṇa* (Motilal Banarsidass, AITM series) | Standard academic translation |
| Translation B | [bhagavata.org](https://bhagavata.org) | **Anand Aadhar**, *Śrīmad Bhāgavatam* (3rd revised ed.) | CC BY-NC-SA 3.0 |

> **Provenance & reuse.** Translation B is Creative Commons (BY-NC-SA 3.0). Translation A is
> Tagare's copyrighted translation as hosted by wisdomlib; it is collected here for
> **personal study and non-commercial NLP research**, matching the terms the Sanskrit e-text
> itself is distributed under. Every record keeps a `source` URL so any excerpt is attributable.
> Do not redistribute the translation fields commercially.

### Output

* `data/bhagavata_purana.json`: full nested corpus (the deliverable)
* `data/bhagavata_purana_flat.jsonl`: one sloka per line, for NLP pipelines

Raw downloads are cached under `data/raw/`, so re-running the notebook is cheap and
does not re-hit the servers.

---
## 1. Environment

In [1]:
# One-time setup (uncomment to install into the active kernel):
# %pip install indic_transliteration requests beautifulsoup4 lxml tqdm

import json
import re
import time
import unicodedata
import collections
import datetime as dt
import pathlib
import hashlib

import requests
from bs4 import BeautifulSoup
from indic_transliteration import sanscript
from indic_transliteration.sanscript import transliterate
from tqdm.auto import tqdm

print("imports ok")

imports ok


In [2]:
# ---- Paths -------------------------------------------------------------
ROOT = pathlib.Path.cwd()
if ROOT.name == "notebooks":              # notebook launched from notebooks/
    ROOT = ROOT.parent

DATA     = ROOT / "data"
RAW      = DATA / "raw"
RAW_WL   = RAW / "wisdomlib"
RAW_BO   = RAW / "bhagavata_org"
for d in (DATA, RAW, RAW_WL, RAW_BO):
    d.mkdir(parents=True, exist_ok=True)

ITX_URL  = "https://sanskritdocuments.org/doc_purana/bhagpur.itx"
ITX_PATH = RAW / "bhagpur.itx"

WL_INDEX_URL = "https://www.wisdomlib.org/hinduism/book/the-bhagavata-purana"
WL_BASE      = "https://www.wisdomlib.org"
BO_CHAPTER   = "https://bhagavata.org/canto{canto}/chapter{chapter}.html"

OUT_JSON  = DATA / "bhagavata_purana.json"
OUT_JSONL = DATA / "bhagavata_purana_flat.jsonl"

# Canonical chapter counts per skandha (used to verify the parse).
CANONICAL_ADHYAYAS = {1: 19, 2: 10, 3: 33, 4: 31, 5: 26, 6: 19,
                      7: 15, 8: 24, 9: 24, 10: 90, 11: 31, 12: 13}
N_ADHYAYAS = sum(CANONICAL_ADHYAYAS.values())   # 335

# Be a polite crawler: bhagavata.org/robots.txt asks for Crawl-delay: 1.
CRAWL_DELAY = 1.0
HEADERS = {"User-Agent": ("BhagavataCorpusBuilder/1.0 (academic NLP research; "
                          "contact: local use only)")}

print(f"ROOT       = {ROOT}")
print(f"adhyayas   = {N_ADHYAYAS}")

ROOT       = /Users/imradhe/Projects/ANLP Project/Shrimad Bhagavata Purana
adhyayas   = 335


In [3]:
SESSION = requests.Session()
SESSION.headers.update(HEADERS)
_last_hit = {"t": 0.0}

_CHARSET_RE = re.compile(b"charset=[\"']?([\\w-]+)", re.I)


def decode_body(resp) -> str:
    """Decode a response without trusting charset guessing.

    UTF-8 is self-validating, so a strict decode that succeeds is strong evidence.
    This matters here: bhagavata.org *declares* windows-1252 while actually serving
    UTF-8, and chardet mis-guesses some wisdomlib pages as a Central-European
    codepage, which silently turns "Śāstra" into mojibake.
    """
    raw = resp.content
    try:
        return raw.decode("utf-8")
    except UnicodeDecodeError:
        pass
    candidates = []
    header = _CHARSET_RE.search(resp.headers.get("content-type", "").encode())
    if header:
        candidates.append(header.group(1).decode())
    meta = _CHARSET_RE.search(raw[:4096])
    if meta:
        candidates.append(meta.group(1).decode())
    candidates += [resp.apparent_encoding or "", "cp1252"]
    for enc in candidates:
        if not enc:
            continue
        try:
            return raw.decode(enc)
        except (UnicodeDecodeError, LookupError):
            continue
    return raw.decode("utf-8", errors="replace")


def fetch(url, cache_path, *, force=False, delay=CRAWL_DELAY):
    """GET `url`, caching the response body at `cache_path`.

    Cached files are reused unless `force=True`, so the notebook can be re-run
    end-to-end without re-hitting either server.
    """
    cache_path = pathlib.Path(cache_path)
    if cache_path.exists() and not force and cache_path.stat().st_size > 0:
        return cache_path.read_text(encoding="utf-8", errors="replace")

    wait = delay - (time.monotonic() - _last_hit["t"])
    if wait > 0:
        time.sleep(wait)

    last_err = None
    for attempt in range(4):
        try:
            r = SESSION.get(url, timeout=45)
            _last_hit["t"] = time.monotonic()
            r.raise_for_status()
            text = decode_body(r)
            cache_path.parent.mkdir(parents=True, exist_ok=True)
            cache_path.write_text(text, encoding="utf-8")
            return text
        except Exception as exc:                      # noqa: BLE001
            last_err = exc
            time.sleep(2 * (attempt + 1))
    raise RuntimeError(f"failed to fetch {url}: {last_err}")


print("fetch() ready")

fetch() ready


---
## 2. Download the Sanskrit source

In [4]:
itx_raw = fetch(ITX_URL, ITX_PATH)
print(f"{ITX_PATH.name}: {len(itx_raw):,} chars, {itx_raw.count(chr(10)):,} lines")
print("sha256:", hashlib.sha256(itx_raw.encode()).hexdigest()[:16])

bhagpur.itx: 1,807,147 chars, 31,703 lines
sha256: f7b3d8240c33c99d


---
## 3. How the source encodes the structure

Every text line carries an 8-digit prefix. It decodes as **`SSCCVVVP`**:

```
0 1 0 1 0 0 1 1
└┬┘ └┬┘ └─┬─┘ │
 │   │    │   └── P  pāda / line within the verse  (1 digit)
 │   │    └────── VVV śloka (verse) number         (3 digits)
 │   └─────────── CC  adhyāya (chapter)            (2 digits)
 └─────────────── SS  skandha (book)               (2 digits)
```

Two special cases:

* **`P == 0`** → not verse text but the **vachana**: a speaker attribution
  (`maitreya uvācha |`) or a prose lead-in (`tatrāyaṃ ślokaḥ`) introducing that verse.
* **Un-prefixed non-empty lines** are soft-wrapped continuations of the preceding
  numbered line. Skandha 5 and parts of 10–12 contain long prose (gadya) passages that
  wrap over several physical lines; these must be re-joined onto their pāda.

The document-level invocation `|| OM namo bhagavate vāsudevāya ||` and closing
`|| OM tatsat ||` sit outside the numbering and are captured separately. Note that a
naive parser would silently glue `|| OM tatsat ||` onto the end of verse 12.13.23.

Two further irregularities in the source have to be handled, or verses silently merge:

* Eight lines in 4.29 carry an inline editorial marker `##vedabase 4.29.1a)##`. They are
  **variant readings from a different edition**, printed under verse numbers that the
  main text also uses. Concatenating them would splice two different verses together, so
  they are stripped out and kept under a `variants` field.
* **24 line-numbers are used twice** (or three times). Most are editorial slips where a
  verse's second half was re-typed with the first half's pāda number, so joining those
  reconstructs the verse correctly. A few (10.62.2–4) collapse consecutive verses under
  one number. Every affected record is flagged in an `anomalies` field rather than being
  quietly normalised.

In [5]:
lines = itx_raw.split("\n")
print("--- numbered verse lines (start of 1.1) ---")
for l in lines[29:35]:
    print("  ", l)

print("\n--- a vachana (pāda 0) ---")
print("  ", next(l for l in lines if re.match(r"^\d{7}0 ", l)))

print("\n--- prose wrap: one numbered line + continuations (5.1.5) ---")
i = next(i for i, l in enumerate(lines) if l.startswith("05010051"))
for l in lines[i:i + 3]:
    print("  ", l[:110] + ("…" if len(l) > 110 else ""))

--- numbered verse lines (start of 1.1) ---
   01010011 janmAdyasya yato.anvayAditaratashchArtheShvabhij~naH svarAT
   01010012 tene brahma hR^idA ya Adikavaye muhyanti yatsUrayaH
   01010013 tejovArimR^idAM yathA vinimayo yatra trisargo.amR^iShA
   01010014 dhAmnA svena sadA nirastakuhakaM satyaM paraM dhImahi
   01010021 dharmaH projjhitakaitavo.atra paramo nirmatsarANAM satAM
   01010022 vedyaM vAstavamatra vastu shivadaM tApatrayonmUlanam

--- a vachana (pāda 0) ---
   01010060 R^iShaya UchuH |

--- prose wrap: one numbered line + continuations (5.1.5) ---
   05010051 bADhamuktaM bhagavata uttamashlokasya shrImachcharaNAravindamakarandarasa Aveshitachetaso
   bhAgavataparamahaMsadayitakathAM ki~nchidantarAyavihatAM svAM shivatamAM padavIM na prAyeNa hinvanti
   05010061 yarhi vAva ha rAjansa rAjaputraH priyavrataH paramabhAgavato nAradasya


---
## 4. Transliteration

The source is ITRANS; we emit Devanāgarī (the requested primary form) and IAST
(convenient for NLP tokenisation and for matching against secondary literature).

In [6]:
def norm_ws(s: str) -> str:
    """Collapse runs of spaces/tabs, preserving newlines."""
    return re.sub(r"[ \t]+", " ", s).strip()


def longest_run(spans):
    """Keep the longest chain of strictly increasing, non-overlapping verse spans.

    Both translation sites contain numbers that look like verse markers but are not:
    an incidental "(2)" inside prose, or a first verse mislabelled with the previous
    chapter's last number (wisdomlib opens 3.6, 10.72 and 11.31 with "51.", "47.."
    and "51.."). Selecting the longest consistent chain discards those outliers
    without a hand-maintained list of exceptions, and degrades gracefully: a bogus
    leading number costs that one verse instead of the whole chapter.

    `spans` is a list of dicts with integer `lo`/`hi`, in document order.
    Returns (kept, dropped) preserving that order.
    """
    n = len(spans)
    if not n:
        return [], []
    best, prev = [1] * n, [-1] * n
    for i in range(n):
        for j in range(i):
            if spans[j]["hi"] < spans[i]["lo"] and best[j] + 1 > best[i]:
                best[i], prev[i] = best[j] + 1, j
    k = max(range(n), key=lambda i: best[i])
    chain = []
    while k != -1:
        chain.append(k)
        k = prev[k]
    keep = set(chain)
    return ([spans[i] for i in sorted(keep)],
            [spans[i] for i in range(n) if i not in keep])


def clamp_overlaps(spans):
    """Stop a grouped range from running past the next verse marker.

    Both sites sometimes mistype a range end. bhagavata.org writes "(22-33)" in 8.6
    where the following "(24)" shows the group is really 22-23, and wisdomlib prints
    overlapping groupings such as "2-6." followed by "3-9." in 6.7. Trusting the
    finer, later marker keeps both blocks and covers every verse, instead of making
    the two compete so whichever loses takes its verses down with it.

    Mutates and returns `spans` (document order).
    """
    for span, nxt in zip(spans, spans[1:]):
        if span["lo"] < nxt["lo"] <= span["hi"]:
            span["hi"] = nxt["lo"] - 1
    return spans


_probe = [{"lo": 51, "hi": 51}, {"lo": 3, "hi": 3}, {"lo": 4, "hi": 5}, {"lo": 6, "hi": 6}]
assert [s["lo"] for s in longest_run(_probe)[0]] == [3, 4, 6]
_probe = clamp_overlaps([{"lo": 22, "hi": 33}, {"lo": 24, "hi": 24}])
assert (_probe[0]["lo"], _probe[0]["hi"]) == (22, 23), _probe
print("longest_run() / clamp_overlaps() verified")


def to_devanagari(itrans: str) -> str:
    return transliterate(itrans, sanscript.ITRANS, sanscript.DEVANAGARI)


def to_iast(itrans: str) -> str:
    return transliterate(itrans, sanscript.ITRANS, sanscript.IAST)


for probe in [
    "janmAdyasya yato.anvayAditaratashchArtheShvabhij~naH svarAT",
    "shrIshuka uvAcha |",
    "|| OM namo bhagavate vAsudevAya ||",
]:
    print(f"ITRANS : {probe}")
    print(f"DEVA   : {to_devanagari(probe)}")
    print(f"IAST   : {to_iast(probe)}\n")

longest_run() / clamp_overlaps() verified
ITRANS : janmAdyasya yato.anvayAditaratashchArtheShvabhij~naH svarAT
DEVA   : जन्माद्यस्य यतोऽन्वयादितरतश्चार्थेष्वभिज्ञः स्वराट्
IAST   : janmādyasya yato'nvayāditarataścārtheṣvabhijñaḥ svarāṭ

ITRANS : shrIshuka uvAcha |
DEVA   : श्रीशुक उवाच ।
IAST   : śrīśuka uvāca |

ITRANS : || OM namo bhagavate vAsudevAya ||
DEVA   : ॥ ॐ नमो भगवते वासुदेवाय ॥
IAST   : || oṃ namo bhagavate vāsudevāya ||



---
## 5. Parse the ITRANS e-text

We walk the file once, keying every pāda by `(skandha, chapter, verse)` and folding
wrapped continuation lines back onto whichever pāda they belong to.

In [7]:
LINE_RE     = re.compile(r"^(\d{2})(\d{2})(\d{3})(\d)\s+(.*)$")
SPEAKER_RE  = re.compile(r"(uvAcha|ovAcha|UchuH|Uchatu|UchatuH)")
# Structural/LaTeX noise emitted by the itrans preprocessor.
DROP_PREFIX = ("%", "\\", "#")
DROP_EXACT  = {"##"}
DROP_STARTS = ("Please send", "Last updated", "http://", "https://")


ANNOT_RE = re.compile(r"##([^#]*)##")


def parse_itx(raw_text):
    """ITRANS e-text -> ordered {(s, c, v): record}.

    Each record holds `padas` ({pada: [line, ...]}), `vachana` (list of lines or
    None), `variants` (other-edition readings printed under the same number) and
    `anomalies` (notes about source irregularities). Returns (verses, meta), where
    meta holds the out-of-band `|| ... ||` lines and anything unplaceable.
    """
    src = raw_text.split("\n")
    try:
        start = next(i for i, l in enumerate(src)
                     if l.startswith(r"\begin{document}")) + 1
    except StopIteration:
        start = 0

    verses = collections.OrderedDict()
    marks  = []           # out-of-band  || ... ||  lines, in document order
    stray  = []           # anything we could not place (expected: empty)
    target = None         # list that wrapped continuation lines append to

    def slot(key):
        return verses.setdefault(key, {"padas": {}, "vachana": None,
                                       "variants": [], "anomalies": []})

    for lineno, rawline in enumerate(src[start:], start=start + 1):
        line = rawline.strip()
        if (not line or line in DROP_EXACT
                or line.startswith(DROP_PREFIX) or line.startswith(DROP_STARTS)):
            continue

        m = LINE_RE.match(line)
        if m:
            s, c, v, p, text = (int(m[1]), int(m[2]), int(m[3]), int(m[4]), m[5].strip())
            notes = [n.strip() for n in ANNOT_RE.findall(text)]
            text = norm_ws(ANNOT_RE.sub(" ", text))
            rec = slot((s, c, v))

            if notes and p > 0:
                # Variant reading from another edition, printed under a number the
                # main text also uses. Keep it, but out of the verse body.
                rec["variants"].append({"pada": p, "note": "; ".join(notes),
                                        "parts": [text], "line": lineno})
                rec["anomalies"].append(f"variant reading present ({'; '.join(notes)})")
                target = rec["variants"][-1]["parts"]
            elif p == 0:
                if rec["vachana"] is None:
                    rec["vachana"] = [text]
                else:
                    rec["anomalies"].append("vachana line numbered more than once")
                    rec["vachana"].append(text)
                target = rec["vachana"]
            else:
                if p in rec["padas"]:
                    rec["anomalies"].append(
                        f"pāda {p} numbered more than once in the source")
                rec["padas"].setdefault(p, []).append(text)
                target = rec["padas"][p]
            continue

        # Un-numbered line.
        if line.startswith("||") and line.endswith("||"):
            # Document-level invocation / colophon: NOT a continuation.
            marks.append({"line": lineno, "itrans": line})
            target = None
            continue

        if target is None:
            stray.append((lineno, line))
            continue
        target.append(norm_ws(ANNOT_RE.sub(" ", line)))

    for rec in verses.values():
        rec["anomalies"] = list(dict.fromkeys(rec["anomalies"]))
    return verses, {"marks": marks, "stray": stray}


verses, meta = parse_itx(itx_raw)

n_anom = sum(1 for r in verses.values() if r["anomalies"])
n_var  = sum(len(r["variants"]) for r in verses.values())
print(f"verse slots parsed : {len(verses):,}")
print(f"vachanas found     : {sum(1 for r in verses.values() if r['vachana']):,}")
print(f"out-of-band marks  : {[m['itrans'] for m in meta['marks']]}")
print(f"variant readings   : {n_var} (kept out of the verse body)")
print(f"records flagged    : {n_anom}")
print(f"unplaceable lines  : {len(meta['stray'])}  (expected 0)")
assert not meta["stray"], meta["stray"][:5]

print("\nflagged records:")
for k, r in verses.items():
    if r["anomalies"]:
        print(f"  {k[0]}.{k[1]}.{k[2]}: {r['anomalies']}")

verse slots parsed : 14,089
vachanas found     : 1,332
out-of-band marks  : ['|| OM namo bhagavate vAsudevAya ||', '|| OM tatsat ||']
variant readings   : 8 (kept out of the verse body)
records flagged    : 18
unplaceable lines  : 0  (expected 0)

flagged records:
  1.7.1: ['pāda 1 numbered more than once in the source']
  3.4.3: ['pāda 1 numbered more than once in the source']
  3.14.49: ['pāda 1 numbered more than once in the source']
  3.32.21: ['pāda 1 numbered more than once in the source']
  4.29.46: ['variant reading present (vedabase 4.29.1a))']
  4.29.47: ['variant reading present (vedabase 4.29.2a))']
  4.29.76: ['variant reading present (vedabase 4.29.1b))']
  4.29.77: ['variant reading present (vedabase 4.29.2b))']
  5.17.17: ['pāda 1 numbered more than once in the source']
  5.26.2: ['pāda 1 numbered more than once in the source']
  6.6.22: ['pāda 2 numbered more than once in the source']
  8.12.47: ['pāda 3 numbered more than once in the source']
  8.16.23: ['pāda 1 numbe

---
## 6. Verify the parse against the canonical structure

If the `SSCCVVVP` reading is correct we should recover exactly 12 skandhas and
**335** adhyāyas, with per-skandha counts matching the received text.

In [8]:
by_adhyaya = collections.OrderedDict()
for (s, c, v) in verses:
    by_adhyaya.setdefault((s, c), []).append(v)

adhyayas_per_skandha = collections.Counter(s for (s, c) in by_adhyaya)

print(f"{'skandha':>8} {'adhyayas':>9} {'canonical':>10}  status")
ok = True
for s in sorted(CANONICAL_ADHYAYAS):
    got, exp = adhyayas_per_skandha[s], CANONICAL_ADHYAYAS[s]
    ok &= got == exp
    print(f"{s:>8} {got:>9} {exp:>10}  {'OK' if got == exp else 'MISMATCH'}")

print(f"\ntotal adhyayas : {len(by_adhyaya)} / {N_ADHYAYAS}   {'OK' if ok else 'FAIL'}")
assert ok and len(by_adhyaya) == N_ADHYAYAS

# Verse numbering should be contiguous 1..n inside each chapter. A handful of
# gaps are genuine to this edition (verses merged or omitted by the editor).
gaps = {f"{s}.{c}": sorted(set(range(1, max(vs) + 1)) - set(vs))
        for (s, c), vs in by_adhyaya.items()
        if set(range(1, max(vs) + 1)) - set(vs)}
print(f"adhyayas with sloka-number gaps: {len(gaps)} -> {gaps}")

pada_patterns = collections.Counter(tuple(sorted(r["padas"])) for r in verses.values())
print("\nmost common pāda layouts (anuṣṭubh is (1, 3)):")
for pat, n in pada_patterns.most_common(6):
    print(f"  {str(pat):<16} {n:>6,}")

 skandha  adhyayas  canonical  status
       1        19         19  OK
       2        10         10  OK
       3        33         33  OK
       4        31         31  OK
       5        26         26  OK
       6        19         19  OK
       7        15         15  OK
       8        24         24  OK
       9        24         24  OK
      10        90         90  OK
      11        31         31  OK
      12        13         13  OK

total adhyayas : 335 / 335   OK
adhyayas with sloka-number gaps: 4 -> {'3.25': [33], '5.9': [9], '8.7': [5], '11.11': [13]}

most common pāda layouts (anuṣṭubh is (1, 3)):
  (1, 3)            7,259
  (1, 2)            5,270
  (1, 2, 3, 4)        842
  (1,)                475
  (1, 3, 5)           143
  (1, 2, 3)            88


---
## 7. Build the sloka records

For each `(skandha, adhyaya, sloka)` we emit:

* `text`: verse text only, as Devanāgarī / IAST / ITRANS, plus a per-pāda `padas` array
* `vachana`: the pāda-0 speaker attribution or prose lead-in, or `null`

One slot in the source (`4.21.45`) carries a trailing `maitreya uvācha |` with no verse
attached, a dangling lead-in at the end of the adhyāya. It is emitted with
`type: "vachana_only"` and `text: null` rather than being dropped or silently merged.

In [9]:
def classify_vachana(itrans: str) -> str:
    return "speaker" if SPEAKER_RE.search(itrans) else "lead_in"


def render(itrans: str) -> dict:
    return {"devanagari": to_devanagari(itrans),
            "iast":       to_iast(itrans),
            "itrans":     itrans}


def build_slokas(verses):
    out = []
    for (s, c, v), rec in verses.items():
        pada_items = []
        for p in sorted(rec["padas"]):
            txt = norm_ws(" ".join(rec["padas"][p]))
            if txt:
                pada_items.append({"pada": p, **render(txt)})

        vachana = None
        if rec["vachana"]:
            vtxt = norm_ws(" ".join(rec["vachana"]))
            vachana = {"type": classify_vachana(vtxt), **render(vtxt)}

        variants = [{"pada": var["pada"], "note": var["note"],
                     **render(norm_ws(" ".join(var["parts"])))}
                    for var in rec["variants"]
                    if norm_ws(" ".join(var["parts"]))]

        if pada_items:
            joined = " ".join(x["itrans"] for x in pada_items)
            text = {
                "devanagari": "\n".join(x["devanagari"] for x in pada_items),
                "iast":       "\n".join(x["iast"] for x in pada_items),
                "itrans":     "\n".join(x["itrans"] for x in pada_items),
                "devanagari_flat": to_devanagari(joined),
                "iast_flat":       to_iast(joined),
                "padas": pada_items,
            }
            kind = "sloka"
        else:
            text, kind = None, "vachana_only"

        record = {
            "id":      f"{s}.{c}.{v}",
            "skandha": s,
            "adhyaya": c,
            "sloka":   v,
            "type":    kind,
            "vachana": vachana,
            "text":    text,
            "translations": {},
        }
        if variants:
            record["variants"] = variants
        if rec["anomalies"]:
            record["anomalies"] = rec["anomalies"]
        out.append(record)
    return out


slokas = build_slokas(verses)
by_id  = {r["id"]: r for r in slokas}

n_sloka   = sum(1 for r in slokas if r["type"] == "sloka")
n_vonly   = sum(1 for r in slokas if r["type"] == "vachana_only")
n_vachana = sum(1 for r in slokas if r["vachana"])
print(f"records      : {len(slokas):,}")
print(f"  slokas     : {n_sloka:,}")
print(f"  vachana-only: {n_vonly}  -> {[r['id'] for r in slokas if r['type'] == 'vachana_only']}")
print(f"vachanas     : {n_vachana:,} "
      f"({sum(1 for r in slokas if r['vachana'] and r['vachana']['type'] == 'speaker'):,} speaker / "
      f"{sum(1 for r in slokas if r['vachana'] and r['vachana']['type'] == 'lead_in'):,} lead-in)")
print(f"variants     : {sum(len(r.get('variants', [])) for r in slokas)} "
      f"on {sum(1 for r in slokas if r.get('variants')):,} records")
print(f"flagged      : {sum(1 for r in slokas if r.get('anomalies')):,} records carry `anomalies`")

records      : 14,089
  slokas     : 14,088
  vachana-only: 1  -> ['4.21.45']
vachanas     : 1,332 (1,323 speaker / 9 lead-in)
variants     : 8 on 4 records
flagged      : 18 records carry `anomalies`


In [10]:
# Spot-check: a verse with a vachana, and a long prose (gadya) passage.
for vid in ("1.1.1", "1.1.6", "5.1.5"):
    r = by_id[vid]
    print(f"=== {vid}  [{r['type']}]")
    if r["vachana"]:
        print(f"  vachana ({r['vachana']['type']}): {r['vachana']['devanagari']}")
    for ln in r["text"]["padas"]:
        txt = ln["devanagari"]
        print(f"  {ln['pada']}: {txt[:96]}{'…' if len(txt) > 96 else ''}")
    print()

=== 1.1.1  [sloka]
  1: जन्माद्यस्य यतोऽन्वयादितरतश्चार्थेष्वभिज्ञः स्वराट्
  2: तेने ब्रह्म हृदा य आदिकवये मुह्यन्ति यत्सूरयः
  3: तेजोवारिमृदां यथा विनिमयो यत्र त्रिसर्गोऽमृषा
  4: धाम्ना स्वेन सदा निरस्तकुहकं सत्यं परं धीमहि

=== 1.1.6  [sloka]
  vachana (speaker): ऋषय ऊचुः ।
  1: त्वया खलु पुराणानि सेतिहासानि चानघ
  3: आख्यातान्यप्यधीतानि धर्मशास्त्राणि यान्युत

=== 5.1.5  [sloka]
  vachana (speaker): श्रीशुक उवाच ।
  1: बाढमुक्तं भगवत उत्तमश्लोकस्य श्रीमच्चरणारविन्दमकरन्दरस आवेशितचेतसो भागवतपरमहंसदयितकथां किञ्चिदन्…



---
## 8. Translation A: G. V. Tagare (via wisdomlib)

wisdomlib hosts Tagare's translation one **chapter per page**, with each verse as a
paragraph beginning `N.` (or `N-M.` where the translator groups verses). We first read the
table of contents to map every `(skandha, chapter)` to its page(s), then fetch each one.

Two things the table of contents makes us handle:

* Six long chapters are **split across lettered sub-pages**: `Chapter 50(a)` … `50(d)`,
  and `10.80` / `12.6` exist *only* as `(a)`/`(b)` with no plain page. Each canonical
  chapter therefore maps to a **list of parts**, whose verses are merged.
* Footnote markers (`[12]`) are stripped from the translation text; the footnote bodies
  live in a separate block that we discard.

Two content quirks are handled by `longest_run` and by continuation absorption:

* Three chapters (**3.6, 10.72, 11.31**) open with the *previous* chapter's last verse
  number (`51.`, `47..`, `51..`), a copy-paste slip in the site's own text. Anchoring on
  the first number seen would reject every subsequent verse and lose the whole chapter,
  so we keep the longest consistent chain and discard the outlier instead.
* A verse is sometimes split so the numbered paragraph carries only a lead-in
  (`3-5. (He observed):`) with the substance in the next, unnumbered paragraph. Unnumbered
  paragraphs are therefore appended to the verse above them.

In [11]:
WL_DOC_RE = re.compile(r'href="(/hinduism/book/the-bhagavata-purana/d/doc(\d+)\.html)"[^>]*>([^<]{0,120})')

# Some wisdomlib pages ship *pre-corrupted* text: the original UTF-8 bytes were
# once decoded as Mac Central European and re-encoded as UTF-8, so "Śāstras"
# arrives as "ŇöńĀstras". The page is valid UTF-8, so no decoding fix can help.
# The damage is a pure round-trip, so it is exactly reversible.
#
# Two guards keep the repair from touching healthy text:
#
#   1. The trigger set contains only characters that CANNOT occur in correct IAST.
#      It deliberately excludes Ā ā Ī ī ē ö õ, which appear in both the corrupted
#      and the correct form. An earlier version included Ā and silently rewrote
#      "Ṛṣabha\u2014Ādinātha" to "Ṛṣabhaсdinātha", because that em dash followed
#      by "Ā" happened to reverse into valid UTF-8 (a Cyrillic letter).
#   2. A reversed run is accepted only if it still looks like Indological
#      transliteration, so an accidental valid-UTF-8 reversal is rejected.
_MOJI_HINT = re.compile(r"[ŇńĄąĆćĎďĘęĚģĶĻĽľłŠŹ]")
_MOJI_RUN  = re.compile(r"[^\x00-\x7F]+")
_MOJI_OK   = re.compile("[\\u00C0-\\u024F\\u1E00-\\u1EFF\\u0900-\\u097F"
                        "\\u2010-\\u2027\\u00A0 \\s]+")


def repair_mojibake(s: str) -> str:
    """Undo a mac_centeuro -> UTF-8 double encoding, leaving clean text alone."""
    if not _MOJI_HINT.search(s):
        return s

    def fix(m):
        run = m.group(0)
        try:
            cand = run.encode("mac_centeuro").decode("utf-8")
        except (UnicodeEncodeError, UnicodeDecodeError):
            return run
        return cand if _MOJI_OK.fullmatch(cand) else run

    return _MOJI_RUN.sub(fix, s)


assert repair_mojibake("ŇöńĀstras") == "Śāstras"
assert repair_mojibake("Ňör ńę Ňöuka said") == "Śr ī Śuka said"
# Healthy text, including the case the earlier guard corrupted, must pass through.
for clean in ("Śrī Śuka, Viṣṇu", "Ṛṣabha\u2014Ādinātha, an incarnation",
              "the Ādi-Nārāyaṇa", "Pañcarātrāgama", "Ārṣa Creation"):
    assert repair_mojibake(clean) == clean, clean
print("repair_mojibake() verified")


WL_CHAP_RE = re.compile("Chapter\\s*(\\d+)\\s*(\\([a-z]\\))?\\s*[-\\u2013\\u2014]\\s*(.*)")


def wisdomlib_index():
    """Map (skandha, chapter) -> {'title', 'parts': [{'doc', 'url', 'part', 'title'}]}.

    A canonical chapter may be served as several lettered sub-pages; all of them
    are kept, in document order, so their verses can be merged.
    """
    html = repair_mojibake(fetch(WL_INDEX_URL, RAW_WL / "_index.html"))
    seen, ordered = set(), []
    for href, doc, label in WL_DOC_RE.findall(html):
        if doc in seen:
            continue
        seen.add(doc)
        ordered.append((href, doc, norm_ws(re.sub(r"\s+", " ", label))))

    index, skandha = collections.OrderedDict(), None
    for href, doc, label in ordered:
        book = re.match(r"Book (\d+)", label)
        if book:
            skandha = int(book.group(1))
            continue
        chap = WL_CHAP_RE.match(label)
        if chap and skandha is not None:
            key = (skandha, int(chap.group(1)))
            entry = index.setdefault(key, {"title": None, "parts": []})
            entry["parts"].append({
                "doc":   doc,
                "url":   WL_BASE + href,
                "part":  (chap.group(2) or "").strip("()") or None,
                "title": chap.group(3).strip(),
            })
    for entry in index.values():
        entry["title"] = " / ".join(dict.fromkeys(p["title"] for p in entry["parts"]))
    return index


WL_INDEX = wisdomlib_index()
multi = {f"{s}.{c}": [p["part"] for p in e["parts"]]
         for (s, c), e in WL_INDEX.items() if len(e["parts"]) > 1}
print(f"wisdomlib chapters indexed: {len(WL_INDEX)} / {N_ADHYAYAS}")
print(f"chapters served as multiple sub-pages: {len(multi)}")
for k, v in multi.items():
    print(f"  {k}: {v}")
assert len(WL_INDEX) == N_ADHYAYAS
print("\n  1.1  ->", WL_INDEX[(1, 1)]["title"])
print(" 10.90 ->", WL_INDEX[(10, 90)]["title"])

repair_mojibake() verified
wisdomlib chapters indexed: 335 / 335
chapters served as multiple sub-pages: 6
  10.50: [None, 'a', 'b', 'c', 'd']
  10.52: [None, 'a', 'b']
  10.59: [None, 'a', 'b', 'c']
  10.60: [None, 'a']
  10.80: ['a', 'b']
  12.6: ['a', 'b']

  1.1  -> Dialogue between Sūta and Śaunaka in the Naimiśa forest
 10.90 -> The Song of Queens: Resume of Kṛṣṇa’s Sports


In [12]:
# Dash characters in these patterns are written as \u2013 / \u2014 escapes rather
# than literals, so the source files stay free of em dashes while the classes still
# match ranges typed with an en or em dash. Verified against all three forms.
VERSE_NUM_RE = re.compile("^(\\d+)\\s*(?:[-\\u2013\\u2014]\\s*(\\d+))?\\s*\\.\\s*(.+)$", re.S)
FOOTNOTE_RE  = re.compile(r"\[\s*\d+\s*\]")
WL_SKIP_RE   = re.compile(
    r"^\[?\s*(Sanskrit text|See notes|back to top|Footnotes|full-text|Let's grow|"
    r"This page|Go directly)", re.I)


def parse_wisdomlib(html):
    """Chapter page -> ({verse_no: {'text', 'ref', 'grouped'}}, dropped_refs).

    Ranges fan out to every verse they cover. A paragraph with no verse number is
    treated as a continuation of the verse above it, because wisdomlib sometimes splits a
    verse so the numbered paragraph holds only a lead-in ("3-5. (He observed):")
    and the substance follows in its own paragraph.
    """
    soup = BeautifulSoup(html, "lxml")
    for t in soup(["script", "style"]):
        t.decompose()
    body = soup.select_one("div.chapter-content")
    if body is None:
        return {}, []
    for d in body.select("div.f"):          # inline footnote bodies
        d.decompose()

    spans, current = [], None
    for p in body.find_all("p"):
        text = norm_ws(p.get_text(" "))
        if not text or WL_SKIP_RE.match(text):
            continue
        m = VERSE_NUM_RE.match(text)
        if m:
            lo = int(m.group(1))
            hi = int(m.group(2)) if m.group(2) else lo
            if hi < lo or hi - lo > 40:     # implausible range -> treat as single
                hi = lo
            current = {"lo": lo, "hi": hi, "parts": [m.group(3)]}
            spans.append(current)
        elif current is not None:
            current["parts"].append(text)

    kept, dropped = longest_run(clamp_overlaps(spans))
    out = {}
    for span in kept:
        payload = norm_ws(FOOTNOTE_RE.sub("", " ".join(span["parts"])))
        payload = norm_ws(re.sub(r"\s+([,.;:?!])", r"\1", payload))
        if not payload:
            continue
        lo, hi = span["lo"], span["hi"]
        ref = f"{lo}" if lo == hi else f"{lo}-{hi}"
        for v in range(lo, hi + 1):
            out.setdefault(v, {"text": payload, "ref": ref, "grouped": lo != hi})
    return out, [f"{d['lo']}-{d['hi']}" for d in dropped]


def wisdomlib_chapter(s, c, entry):
    """Fetch every sub-page of a chapter and merge their verses."""
    verses, dropped = {}, []
    for part in entry["parts"]:
        suffix = f".{part['part']}" if part["part"] else ""
        html = repair_mojibake(fetch(part["url"], RAW_WL / f"{s}.{c}{suffix}.html"))
        vs, drop = parse_wisdomlib(html)
        dropped += drop
        for v, tr in vs.items():
            verses.setdefault(v, {**tr, "source": part["url"]})
    return verses, dropped


# Smoke-test before crawling all 335 chapters: a plain chapter and a split one.
# A plain chapter, a split one, and one whose first verse the site mislabels.
for probe_key in [(1, 1), (12, 6), (3, 6), (11, 31)]:
    got, drop = wisdomlib_chapter(*probe_key, WL_INDEX[probe_key])
    have = len([v for v in by_adhyaya[probe_key]
                if by_id[f"{probe_key[0]}.{probe_key[1]}.{v}"]["type"] == "sloka"])
    print(f"{probe_key[0]}.{probe_key[1]}: {len(got):>3} translated / {have:>3} slokas "
          f"({len(WL_INDEX[probe_key]['parts'])} page(s))  discarded markers: {drop}")
print("\n1.1 v1:", wisdomlib_chapter(1, 1, WL_INDEX[(1, 1)])[0][1]["text"][:170], "…")

1.1:  23 translated /  23 slokas (1 page(s))  discarded markers: []
12.6:  80 translated /  80 slokas (2 page(s))  discarded markers: []
3.6:  38 translated /  40 slokas (1 page(s))  discarded markers: ['51-51']
11.31:  24 translated /  28 slokas (1 page(s))  discarded markers: ['51-51', '52-52']

1.1 v1: Let us meditate upon the Supreme Spirit who is real; from whom emanate the creation etc. (i.e. creation, preservation and destruction) of this (universe), (as can be infe …


In [13]:
tagare = {}
for (s, c), entry in tqdm(sorted(WL_INDEX.items()), desc="wisdomlib (Tagare)"):
    verses, dropped = wisdomlib_chapter(s, c, entry)
    tagare[(s, c)] = {
        "meta": {"title": entry["title"],
                 "urls": [p["url"] for p in entry["parts"]],
                 "discarded_markers": dropped},
        "verses": verses,
    }

got = sum(len(v["verses"]) for v in tagare.values())
empty = [f"{s}.{c}" for (s, c), v in tagare.items() if not v["verses"]]
noisy = {f"{s}.{c}": v["meta"]["discarded_markers"]
         for (s, c), v in tagare.items() if v["meta"]["discarded_markers"]}
print(f"\nchapters fetched : {len(tagare)}")
print(f"verse entries    : {got:,}")
print(f"empty chapters   : {len(empty)} {empty[:10]}")
print(f"chapters with discarded markers: {len(noisy)}")
for k, v in list(noisy.items())[:12]:
    print(f"  {k}: {v}")

wisdomlib (Tagare):   0%|          | 0/335 [00:00<?, ?it/s]


chapters fetched : 335
verse entries    : 14,022
empty chapters   : 0 []
chapters with discarded markers: 39
  2.2: ['25-25']
  2.5: ['9-9']
  3.6: ['51-51']
  3.11: ['23-23']
  4.10: ['43-43', '44-44', '45-45']
  6.6: ['1-1']
  7.11: ['1-1']
  8.1: ['11-11']
  8.3: ['22-24']
  9.4: ['11-11']
  9.8: ['11-11', '1-1', '2-2', '1-1', '1-1']
  10.13: ['40-40']


---
## 9. Translation B: Anand Aadhar (bhagavata.org)

One page per chapter at `bhagavata.org/canto{N}/chapter{M}.html`. Verses are marked
inline as `(N)` or `(N-M)`, anchored to `#Text_N`.

Two quirks to handle:

* Many verses open with a **styled drop-cap**: `<b>L</b>` followed by `et there be…` in
  a separate `<font>`. Joining text with a space would produce `L et there be`, and
  patching that with a regex would corrupt verses that genuinely begin with the word
  `I` (e.g. 12.13.23, `I offer my obeisances`). We therefore extract text using real
  **HTML inline semantics**: adjacent inline tags concatenate with no separator, while
  block-level tags are broken with newlines. `<b>L</b><font>et</font>` → `Let`, and a
  genuine `I offer` keeps its own whitespace text node.
* After the last verse the page repeats the whole chapter in *previous editions* plus
  Vedabase text. We truncate at that boundary so only the current translation is captured.
* Parenthesised numbers also occur incidentally inside the translation prose, so marker
  selection goes through the same `longest_run` filter used for wisdomlib.

In [14]:
# See the note on dash escapes in the wisdomlib section above.
BO_VERSE_RE = re.compile("\\((\\d+)\\s*(?:[-\\u2013\\u2014]\\s*(\\d+))?\\)")
BO_CUT_RE   = re.compile(
    r"(Read the inspiration to this chapter"
    r"|Previous Aadhar edition"
    r"|The text and audio are offered"
    r"|Production: Filognostic)", re.I)
# Non-greedy and anchored on the first verse marker, so the title match ends
# exactly where the translation begins.
BO_TITLE_RE = re.compile(r"Chapter\s*\d+\s*:\s*(.{0,160}?)\s*(?=\(\d)")
BLOCK_TAGS = ["p", "div", "tr", "li", "h1", "h2", "h3", "h4", "h5",
              "blockquote", "table", "td", "center"]


def html_text(node) -> str:
    """Extract text using HTML inline semantics.

    Adjacent *inline* elements concatenate with no separator, which is how a
    browser renders them, so a styled drop-cap letter re-joins its word. Block
    elements and <br> are separated by newlines so words never run together
    across a visual line break.
    """
    for br in node.find_all("br"):
        br.replace_with("\n")
    for tag in node.find_all(BLOCK_TAGS):
        tag.insert_before("\n")
        tag.insert_after("\n")
    return node.get_text("")


def parse_bhagavata_org(html):
    """Chapter page -> (chapter_title, {verse_no: {'text', 'ref', 'grouped'}})."""
    soup = BeautifulSoup(html, "lxml")
    for t in soup(["script", "style"]):
        t.decompose()
    text = norm_ws(re.sub(r"\s+", " ", html_text(soup)))

    title_m = BO_TITLE_RE.search(text)
    title = norm_ws(title_m.group(1)) if title_m else None

    # Begin after the chapter heading, so parenthesised numbers in the page
    # furniture cannot be mistaken for verse markers.
    body = text[title_m.end():] if title_m else text
    cut = BO_CUT_RE.search(body)
    if cut:
        body = body[:cut.start()]

    # Translation prose contains incidental parentheses, so a bare "(2)" mid-sentence
    # is commentary rather than a verse marker. Keep the longest consistent chain.
    spans = []
    for m in BO_VERSE_RE.finditer(body):
        lo = int(m.group(1))
        hi = int(m.group(2)) if m.group(2) else lo
        if hi < lo or hi - lo > 40:
            continue
        spans.append({"lo": lo, "hi": hi, "start": m.start(), "end": m.end()})
    kept, dropped = longest_run(clamp_overlaps(spans))

    out = {}
    for i, span in enumerate(kept):
        # Run to the next *accepted* marker, so a discarded incidental number does
        # not truncate the verse it sits inside.
        end = kept[i + 1]["start"] if i + 1 < len(kept) else len(body)
        chunk = norm_ws(re.sub(r"\s+([,.;:?!])", r"\1",
                               norm_ws(body[span["end"]:end])))
        if not chunk:
            continue
        lo, hi = span["lo"], span["hi"]
        ref = f"{lo}" if lo == hi else f"{lo}-{hi}"
        for v in range(lo, hi + 1):
            out.setdefault(v, {"text": chunk, "ref": ref, "grouped": lo != hi})
    return title, out, [f"{d['lo']}-{d['hi']}" for d in dropped]


_t, _p, _d = parse_bhagavata_org(fetch(BO_CHAPTER.format(canto=1, chapter=1),
                                       RAW_BO / "1.1.html"))
print(f"title: {_t}  discarded markers: {_d}")
print(f"1.1 verses parsed: {len(_p)}  (Sanskrit has {len(by_adhyaya[(1, 1)])})")
print("v1:", _p[1]["text"][:180], "…")

title: Questions by the Sages  discarded markers: []
1.1 verses parsed: 23  (Sanskrit has 23)
v1: Let there be the salutation of the original appearance of Him, Vâsudeva, the Fortunate One, from whom, being present here and in the beyond, for the purpose of recollection and ful …


In [15]:
aadhar = {}
targets = [(s, c) for s, n in sorted(CANONICAL_ADHYAYAS.items())
           for c in range(1, n + 1)]

for s, c in tqdm(targets, desc="bhagavata.org (Aadhar)"):
    url = BO_CHAPTER.format(canto=s, chapter=c)
    html = fetch(url, RAW_BO / f"{s}.{c}.html")
    title, vs, dropped = parse_bhagavata_org(html)
    aadhar[(s, c)] = {
        "meta": {"title": title, "urls": [url], "discarded_markers": dropped},
        "verses": {v: {**tr, "source": url} for v, tr in vs.items()},
    }

got = sum(len(v["verses"]) for v in aadhar.values())
empty = [f"{s}.{c}" for (s, c), v in aadhar.items() if not v["verses"]]
noisy = {f"{s}.{c}": v["meta"]["discarded_markers"]
         for (s, c), v in aadhar.items() if v["meta"]["discarded_markers"]}
print(f"\nchapters fetched : {len(aadhar)}")
print(f"verse entries    : {got:,}")
print(f"empty chapters   : {len(empty)} {empty[:10]}")
print(f"chapters with discarded markers: {len(noisy)}")
for k, v in list(noisy.items())[:12]:
    print(f"  {k}: {v}")

bhagavata.org (Aadhar):   0%|          | 0/335 [00:00<?, ?it/s]


chapters fetched : 335
verse entries    : 14,087
empty chapters   : 0 []
chapters with discarded markers: 1
  3.10: ['1-1', '2-2', '3-3', '4-4', '5-5', '6-6', '7-7', '8-8']


---
## 10. Align translations to the Sanskrit

Both translations follow the same canonical `skandha.chapter.verse` reference, so we join
on it directly. We do **not** re-index or shift anything to force a match: where a source
omits or groups verses differently we record that honestly (`ref`, `grouped`) and report
coverage, so downstream users can see exactly how well each chapter aligns.

In [16]:
SOURCE_META = {
    "tagare": {
        "translator": "G. V. Tagare",
        "edition": ("The Bhāgavata Purāṇa, Ancient Indian Tradition & Mythology series, "
                    "Motilal Banarsidass"),
        "host": "wisdomlib.org",
        "rights": ("Copyright of the translator/publisher; collected here for personal "
                   "study and non-commercial research only."),
    },
    "anand_aadhar": {
        "translator": "Anand Aadhar (Hans van Belle)",
        "edition": "Śrīmad Bhāgavatam, third revised edition",
        "host": "bhagavata.org",
        "rights": "Creative Commons Attribution-NonCommercial-ShareAlike 3.0 Unported",
    },
}


def attach(store, key):
    hits, orphans = 0, []
    for (s, c), payload in store.items():
        for v, tr in payload["verses"].items():
            rec = by_id.get(f"{s}.{c}.{v}")
            if rec is None:            # translation numbers a verse the e-text lacks
                orphans.append(f"{s}.{c}.{v}")
                continue
            rec["translations"][key] = {
                "text":    tr["text"],
                "ref":     tr["ref"],
                "grouped": tr["grouped"],
                "source":  tr.get("source") or payload["meta"]["urls"][0],
            }
            hits += 1
    return hits, orphans


n_t, orphans_t = attach(tagare, "tagare")
n_a, orphans_a = attach(aadhar, "anand_aadhar")
print(f"translated verses with no matching sloka. Tagare: {len(orphans_t)} "
      f"{orphans_t[:6]}, Aadhar: {len(orphans_a)} {orphans_a[:6]}\n")

target = n_sloka
print(f"slokas in corpus            : {target:,}")
print(f"Tagare translations attached: {n_t:,}  ({n_t / target:.1%})")
print(f"Aadhar translations attached: {n_a:,}  ({n_a / target:.1%})")
print(f"slokas with BOTH            : "
      f"{sum(1 for r in slokas if len(r['translations']) == 2):,}")
print(f"slokas with NEITHER         : "
      f"{sum(1 for r in slokas if r['type'] == 'sloka' and not r['translations']):,}")

translated verses with no matching sloka. Tagare: 77 ['1.6.40', '1.6.41', '1.6.42', '3.10.30', '3.10.31', '3.25.33'], Aadhar: 20 ['2.6.46', '2.9.46', '3.5.51', '3.10.30', '3.11.42', '3.12.57']

slokas in corpus            : 14,088
Tagare translations attached: 13,945  (99.0%)
Aadhar translations attached: 14,067  (99.9%)
slokas with BOTH            : 13,932
slokas with NEITHER         : 9


In [17]:
# Which chapters align worst? Useful for spotting a source that numbers differently.
rows = []
for (s, c), vs in by_adhyaya.items():
    ids = [f"{s}.{c}.{v}" for v in vs]
    real = [i for i in ids if by_id[i]["type"] == "sloka"]
    if not real:
        continue
    t = sum(1 for i in real if "tagare" in by_id[i]["translations"])
    a = sum(1 for i in real if "anand_aadhar" in by_id[i]["translations"])
    rows.append((t / len(real), a / len(real), f"{s}.{c}", len(real), t, a))

rows.sort(key=lambda r: min(r[0], r[1]))
print(f"{'adhyaya':>8} {'slokas':>7} {'Tagare':>16} {'Aadhar':>16}")
for tc, ac, ch, n, t, a in rows[:15]:
    print(f"{ch:>8} {n:>7} {t:>7} ({tc:>6.1%}) {a:>7} ({ac:>6.1%})")

full_t = sum(1 for r in rows if r[0] == 1.0)
full_a = sum(1 for r in rows if r[1] == 1.0)
print(f"\nadhyayas fully covered: Tagare {full_t}/{len(rows)}, Aadhar {full_a}/{len(rows)}")

 adhyaya  slokas           Tagare           Aadhar
     9.7      27      23 ( 85.2%)      26 ( 96.3%)
   11.31      28      24 ( 85.7%)      28 (100.0%)
     7.3      38      34 ( 89.5%)      38 (100.0%)
    8.10      57      51 ( 89.5%)      57 (100.0%)
   10.32      22      20 ( 90.9%)      22 (100.0%)
    9.10      56      51 ( 91.1%)      55 ( 98.2%)
   11.19      45      41 ( 91.1%)      45 (100.0%)
     8.8      47      43 ( 91.5%)      46 ( 97.9%)
   11.10      37      34 ( 91.9%)      37 (100.0%)
    12.7      25      23 ( 92.0%)      25 (100.0%)
     5.7      14      13 ( 92.9%)      14 (100.0%)
   11.22      61      57 ( 93.4%)      61 (100.0%)
   10.82      48      45 ( 93.8%)      48 (100.0%)
     8.5      50      47 ( 94.0%)      50 (100.0%)
   10.65      34      32 ( 94.1%)      34 (100.0%)

adhyayas fully covered: Tagare 254/335, Aadhar 315/335


---
## 10b. The per-skandha edition: apparatus and the Māhātmya

Everything so far comes from the **combined** `bhagpur.itx`, which contains only
8-digit-prefixed verse lines. sanskritdocuments also publishes the text as **one file per
skandha** (`bhagpur-01.itx` … `bhagpur-12.itx`, with 10 split into `10a`/`10b`), plus
`bhagpur-00-mahatmyam.itx`. Those are a different, more recently proofread edition, and they
carry material the combined file simply does not have:

* `\chapter{.. OM namo bhagavate vAsudevAya ..}` the invocation
* `\section{.. prathamaskandhaH ..}` and `\section{.. prathamo.adhyAyaH ..}` the **structural
  headings**, exactly the `atha prathamo'dhyāyaḥ` style vachanas worth tagging separately
* a **maṅgala block** before skandha 1's first adhyāya: seven prayer verses and a nyāsa
  sequence (`guṃ gurubhyo namaḥ | gaṃ gaṇapataye namaḥ | ...`). This is a reciter's
  preliminary prayer, not Purāṇa text. Two of its verses are in fact 12.13.22 and 12.13.23
  repeated.
* an `iti ... adhyāyaḥ` **colophon** closing each adhyāya, naming its subject
* the **Bhāgavata Māhātmya**, 6 adhyāyas whose own colophon places it in the
  *Padma Purāṇa, Uttarakhaṇḍa*, so it frames the Bhāgavata rather than belonging to it

### What is taken from where, and why

Verse text stays on the combined edition, which is what the 12 / 335 / 14,088 structure was
validated against. The two editions **disagree slightly on verse splitting** (per-skandha
totals run 805 vs 813 for skandha 1, 567 vs 565 for skandha 12), so rebuilding verses from
the per-skandha files would silently shift verse numbers and invalidate both the translation
alignment and the chandas run. Only apparatus, which attaches at adhyāya level and does not
depend on verse alignment, is taken from them.

The Māhātmya is different: it is absent from the combined file altogether, so it is parsed
in full from its own file and kept as a **separate top-level `mahatmya` section** rather than
being folded in as a thirteenth skandha. That keeps the canonical 12 / 335 assertion intact.

### Source defects found

Colophons are a complete adhyāya index except for three omissions: **4.2**, **7.13** and
**10.25**. Skandha 7 also writes adhyāya 1's colophon as `(1)` rather than `|| 1||`, which the
parser accepts. Skandha 10's two files number their adhyāyas continuously, 1 to 49 in `10a`
and 50 to 90 in `10b`, so no offset is needed. Missing colophons are recorded as `null` rather
than guessed at, and §12 asserts the exact set so a new omission would fail the run.

In [18]:
PER_SKANDHA_FILES = {0: ["00-mahatmyam"], 10: ["10a", "10b"]}
for _s in range(1, 13):
    PER_SKANDHA_FILES.setdefault(_s, [f"{_s:02d}"])

RAW_PS = RAW / "per_skandha"
RAW_PS.mkdir(parents=True, exist_ok=True)
PS_URL = "https://sanskritdocuments.org/doc_purana/bhagpur-{stem}.itx"

per_skandha_raw = {}
for _s, stems in sorted(PER_SKANDHA_FILES.items()):
    for stem in stems:
        per_skandha_raw[stem] = fetch(PS_URL.format(stem=stem),
                                      RAW_PS / f"bhagpur-{stem}.itx")
print(f"per-skandha sources fetched: {len(per_skandha_raw)}")
for stem, txt in list(per_skandha_raw.items())[:3]:
    print(f"  bhagpur-{stem}.itx  {len(txt):>8,} chars")

per-skandha sources fetched: 14
  bhagpur-00-mahatmyam.itx    59,019 chars
  bhagpur-01.itx   100,680 chars
  bhagpur-02.itx    51,535 chars


In [19]:
SEC_RE   = re.compile(r"^\\section\{\.\.\s*(.*?)\s*\.\.\}")
CHAP_RE  = re.compile(r"^\\chapter\{\.\.\s*(.*?)\s*\.\.\}")
# Verse and colophon numbers are written  || N||  and, in two colophons, (N).
PS_NUM_RE   = re.compile(r"(?:\|\|\s*(\d+)\s*\|\||\((\d+)\))\s*$")
PS_COLO_RE  = re.compile(r"adhyAyaH\s*(?:\|\|\s*\d+\s*\|\||\(\d+\))\s*$")
PS_DROP = ("%", "#", "\\documentstyle", "\\portraitwide", "\\parindent", "\\let",
           "\\pagenumbering", "\\def", "\\begin", "\\end", "\\medskip", "\\engtitle",
           "\\itxtitle", "\\hrule", "\\obeylines", "Please send", "Last updated", "http")


def ps_clean(text):
    """Strip itrans word-break markers and cell separators, collapse whitespace."""
    return norm_ws(re.sub(r"\s+", " ", text.replace("\\-", "").replace("##", "")))


def parse_per_skandha(raw_text):
    """Per-skandha / Māhātmya ITX -> (invocation, mangala, {adhyaya: {...}}).

    A numbered line closes an adhyāya when the accumulated text ends in an
    `iti ... adhyāyaḥ` colophon, and is otherwise a verse. Colophons span two
    source lines, so the test has to run against the buffer, not the line.
    """
    invocation, mangala = None, []
    adhyayas = collections.OrderedDict()
    buf, heading, started, pending = [], None, False, {}

    for raw in raw_text.split("\n"):
        line = raw.strip()
        if not line or line == "\\iti" or line.startswith(PS_DROP):
            continue

        m = CHAP_RE.match(line)
        if m:
            invocation = invocation or ps_clean(m.group(1))
            continue
        m = SEC_RE.match(line)
        if m:
            title = ps_clean(m.group(1))
            if "adhyAyaH" in title:
                heading, started = title, True
            continue

        buf.append(line)
        joined = ps_clean(" ".join(buf))
        num = PS_NUM_RE.search(line)
        # Before the first adhyāya nothing is numbered, so a bare `||` ends a unit.
        if not num and not (not started and line.endswith("||")):
            continue

        body = norm_ws(re.sub(r"(?:\|\|\s*\d+\s*\|\||\(\d+\))$", "", joined))
        if not started:
            mangala.append(body)
            buf = []
            continue

        buf = []
        if PS_COLO_RE.search(joined):
            n = int(num.group(1) or num.group(2))
            adhyayas[n] = {"heading": heading, "colophon": body, "verses": pending}
            pending, heading = {}, None
        else:
            pending[int(num.group(1) or num.group(2))] = body

    if pending:
        adhyayas["unclosed"] = {"heading": heading, "colophon": None, "verses": pending}
    return invocation, mangala, adhyayas


# Parse every file and report against the canonical adhyāya counts.
parsed_ps = {stem: parse_per_skandha(txt) for stem, txt in per_skandha_raw.items()}
print(f"{'file':<14} {'colophons':>9} {'verses':>7} {'mangala':>8}  note")
for stem, (inv, mang, adh) in parsed_ps.items():
    ints = sorted(k for k in adh if isinstance(k, int))
    nv = sum(len(v["verses"]) for v in adh.values())
    note = ""
    if stem.isdigit():
        exp = CANONICAL_ADHYAYAS[int(stem)]
        missing = [i for i in range(1, exp + 1) if i not in ints]
        note = "OK" if not missing else f"source omits colophon for adhyaya {missing}"
    print(f"bhagpur-{stem:<6} {len(ints):>9} {nv:>7} {len(mang):>8}  {note}")

file           colophons  verses  mangala  note
bhagpur-00-mahatmyam         6     504        1  
bhagpur-01            19     805        9  OK
bhagpur-02            10     389        0  OK
bhagpur-03            33    1411        0  OK
bhagpur-04            30    1420        0  source omits colophon for adhyaya [2]
bhagpur-05            26     667        0  OK
bhagpur-06            19     851        0  OK
bhagpur-07            14     708        0  source omits colophon for adhyaya [13]
bhagpur-08            24     931        0  OK
bhagpur-09            24     964        0  OK
bhagpur-10a           48    1988        0  
bhagpur-10b           41    1931        0  
bhagpur-11            31    1367        0  OK
bhagpur-12            13     567        0  OK


In [20]:
# ---- Apparatus for the 12 skandhas: headings, colophons, mangala -------------
apparatus = {}            # (skandha, adhyaya) -> {heading, colophon}
skandha_front = {}        # skandha -> {invocation, mangala}

for s, stems in sorted(PER_SKANDHA_FILES.items()):
    if s == 0:
        continue
    inv_all, mang_all = None, []
    for stem in stems:
        inv, mang, adh = parsed_ps[stem]
        inv_all = inv_all or inv
        mang_all += mang
        # 10a and 10b number adhyayas continuously (1..49 then 50..90), so the
        # numbers can be used directly with no offset.
        for n in sorted(k for k in adh if isinstance(k, int)):
            apparatus[(s, n)] = {"heading": adh[n]["heading"],
                                 "colophon": adh[n]["colophon"]}
    skandha_front[s] = {"invocation": inv_all, "mangala": mang_all}

have = sum(1 for k in apparatus if apparatus[k]["colophon"])
want = sum(CANONICAL_ADHYAYAS.values())
print(f"adhyayas with a colophon : {have} / {want}")
missing = [f"{s}.{c}" for s, n in sorted(CANONICAL_ADHYAYAS.items())
           for c in range(1, n + 1) if not apparatus.get((s, c), {}).get("colophon")]
print(f"missing                  : {missing}")
print(f"skandhas with a mangala block: "
      f"{[s for s, v in skandha_front.items() if v['mangala']]}")
print(f"\nsample, skandha 1 adhyaya 1:")
print("  heading :", to_devanagari(apparatus[(1, 1)]['heading'] or ''))
print("  colophon:", to_devanagari(apparatus[(1, 1)]['colophon'] or '')[:110])

adhyayas with a colophon : 332 / 335
missing                  : ['4.2', '7.13', '10.25']
skandhas with a mangala block: [1]

sample, skandha 1 adhyaya 1:
  heading : प्रथमोऽध्यायः
  colophon: इति श्रीमद्भागवते महापुराणे पारमहंस्यां संहितायां प्रथमस्कन्धे नैमिषीयोपाख्याने प्रथमोऽध्यायः


In [21]:
# ---- Mahatmya verses --------------------------------------------------------
mh_inv, mh_mangala, mh_adhyayas = parsed_ps["00-mahatmyam"]
MH_COUNT = {k: len(v["verses"]) for k, v in mh_adhyayas.items()}
print("Mahatmya adhyayas and verse counts:", MH_COUNT)
print("total verses:", sum(MH_COUNT.values()))
print("invocation  :", to_devanagari(mh_inv or ""))
print("mangala     :", to_devanagari(mh_mangala[0]) if mh_mangala else None)
print("colophon a1 :", to_devanagari(mh_adhyayas[1]["colophon"])[:120])
assert sorted(MH_COUNT) == [1, 2, 3, 4, 5, 6], MH_COUNT

Mahatmya adhyayas and verse counts: {1: 80, 2: 76, 3: 74, 4: 81, 5: 90, 6: 103}
total verses: 504
invocation  : ॐ नमो भगवते वासुदेवाय
mangala     : कृष्णं नारायणं वन्दे कृष्णं वन्दे व्रजप्रियम् । कृष्णं द्वैपायनं वन्दे कृष्णं वन्दे पृथासुतम् ॥
colophon a1 : इति श्रीपद्मपुराणे उत्तरखण्डे श्रीमद्भागवतमाहात्म्ये भक्तिनारदसमागमो नाम प्रथमोऽध्यायः


### Tagare's translation of the Māhātmya

wisdomlib carries it as six chapters that sit **before** the first `Book N` marker in the
table of contents, which is exactly why the skandha index built in §8 skipped them. They are
picked up here by their own labels.

In [22]:
def wisdomlib_mahatmya_index():
    """The six Māhātmya chapters, which precede the first `Book N` marker."""
    html = repair_mojibake(fetch(WL_INDEX_URL, RAW_WL / "_index.html"))
    seen, ordered = set(), []
    for href, doc, label in WL_DOC_RE.findall(html):
        if doc in seen:
            continue
        seen.add(doc)
        ordered.append((href, doc, norm_ws(re.sub(r"\s+", " ", label))))

    out = {}
    for href, doc, label in ordered:
        if label.startswith("Book "):
            break                                  # the Purana proper starts here
        m = WL_CHAP_RE.match(label)
        if m:
            out[int(m.group(1))] = {"url": WL_BASE + href, "title": m.group(3).strip()}
    return out


MH_WL = wisdomlib_mahatmya_index()
print(f"Mahatmya chapters on wisdomlib: {sorted(MH_WL)}")
for k in sorted(MH_WL):
    print(f"  {k}: {MH_WL[k]['title']}")
assert sorted(MH_WL) == [1, 2, 3, 4, 5, 6], sorted(MH_WL)

mh_tagare = {}
for c, info in tqdm(sorted(MH_WL.items()), desc="wisdomlib (Mahatmya)"):
    html = repair_mojibake(fetch(info["url"], RAW_WL / f"mahatmya.{c}.html"))
    verses, dropped = parse_wisdomlib(html)
    mh_tagare[c] = {"meta": {**info, "discarded_markers": dropped}, "verses": verses}
    print(f"  adhyaya {c}: {len(verses):>3} translated / {MH_COUNT[c]:>3} verses"
          f"{'  discarded ' + str(dropped) if dropped else ''}")

Mahatmya chapters on wisdomlib: [1, 2, 3, 4, 5, 6]
  1: Nārada Meets Bhakti (Devotion in a Human Form)
  2: Conversation between Nārada and the Kumaras
  3: Removal of Bhakti’s Miseries
  4: Salvation to a Brāhmaṇa—Ātmadeva
  5: Gokarṇa attains to Goloka
  6: The Procedure of Listening to the Bhāgavata


wisdomlib (Mahatmya):   0%|          | 0/6 [00:00<?, ?it/s]

  adhyaya 1:  80 translated /  80 verses


  adhyaya 2:  75 translated /  76 verses
  adhyaya 3:  72 translated /  74 verses  discarded ['10-10']
  adhyaya 4:  80 translated /  81 verses
  adhyaya 5:  89 translated /  90 verses
  adhyaya 6: 105 translated / 103 verses  discarded ['831-831']


In [23]:
# ---- Build the Mahatmya section --------------------------------------------
def render_ps(itrans_text):
    return {"devanagari": to_devanagari(itrans_text),
            "iast": to_iast(itrans_text),
            "itrans": itrans_text}


mahatmya_adhyayas = []
mh_records = []
for c in sorted(k for k in mh_adhyayas if isinstance(k, int)):
    entry = mh_adhyayas[c]
    # Deliberately not named `slokas`: that global holds the 14,089 Purana records
    # and the QA in section 12 iterates it. Shadowing it here silently reduced
    # those checks to whichever Mahatmya adhyaya was parsed last.
    mh_slokas = []
    for v in sorted(entry["verses"]):
        text_itrans = entry["verses"][v]
        rec = {
            "id": f"M.{c}.{v}",
            "part": "mahatmya",
            "skandha": 0,
            "adhyaya": c,
            "sloka": v,
            "type": "sloka",
            "vachana": None,
            "text": {**render_ps(text_itrans),
                     "devanagari_flat": to_devanagari(text_itrans),
                     "iast_flat": to_iast(text_itrans),
                     "padas": [{"pada": 1, **render_ps(text_itrans)}]},
            "translations": {},
        }
        tr = mh_tagare[c]["verses"].get(v)
        if tr:
            rec["translations"]["tagare"] = {
                "text": tr["text"], "ref": tr["ref"], "grouped": tr["grouped"],
                "source": mh_tagare[c]["meta"]["url"],
            }
        mh_slokas.append(rec)
        mh_records.append(rec)

    mahatmya_adhyayas.append({
        "adhyaya": c,
        "heading": render_ps(entry["heading"]) if entry["heading"] else None,
        "colophon": render_ps(entry["colophon"]) if entry["colophon"] else None,
        "titles": {"tagare": mh_tagare[c]["meta"]["title"]},
        "sloka_count": len(mh_slokas),
        "slokas": mh_slokas,
    })

n_mh = len(mh_records)
n_mh_tr = sum(1 for r in mh_records if r["translations"])
print(f"Mahatmya slokas        : {n_mh}")
print(f"  with Tagare          : {n_mh_tr} ({n_mh_tr / n_mh:.1%})")
print(f"  adhyayas             : {len(mahatmya_adhyayas)}")

Mahatmya slokas        : 504
  with Tagare          : 498 (98.8%)
  adhyayas             : 6


---
## 11. Assemble the nested corpus

In [24]:
def adhyaya_titles(s, c):
    return {
        "tagare":       (tagare.get((s, c), {}).get("meta") or {}).get("title"),
        "anand_aadhar": (aadhar.get((s, c), {}).get("meta") or {}).get("title"),
    }


SKANDHA_NAMES = {
    1: "Creation / Questions by the Sages",
    2: "The Cosmic Manifestation",
    3: "The Status Quo",
    4: "The Creation of the Fourth Order",
    5: "The Creative Impetus",
    6: "Prescribed Duties for Mankind",
    7: "The Science of God",
    8: "Withdrawal of the Cosmic Creations",
    9: "Liberation",
    10: "The Summum Bonum",
    11: "General History",
    12: "The Age of Deterioration",
}

marks = meta["marks"]
invocation = render(marks[0]["itrans"]) if marks else None
colophon   = render(marks[-1]["itrans"]) if len(marks) > 1 else None

skandhas = []
for s in sorted(CANONICAL_ADHYAYAS):
    adhyayas = []
    for c in range(1, CANONICAL_ADHYAYAS[s] + 1):
        recs = [by_id[f"{s}.{c}.{v}"] for v in by_adhyaya[(s, c)]]
        app = apparatus.get((s, c), {})
        adhyayas.append({
            "skandha": s,
            "adhyaya": c,
            "titles": adhyaya_titles(s, c),
            # Structural vachanas from the per-skandha edition, kept apart from
            # the verse text. `null` where that edition omits them.
            "heading": render_ps(app["heading"]) if app.get("heading") else None,
            "colophon": render_ps(app["colophon"]) if app.get("colophon") else None,
            "sloka_count": sum(1 for r in recs if r["type"] == "sloka"),
            "slokas": recs,
        })
    front = skandha_front.get(s, {})
    skandhas.append({
        "skandha": s,
        "name": SKANDHA_NAMES[s],
        "invocation": render_ps(front["invocation"]) if front.get("invocation") else None,
        # Reciter's preliminary prayer, present only before skandha 1. Not Purana text.
        "mangala": [render_ps(m) for m in front.get("mangala", [])] or None,
        "adhyaya_count": len(adhyayas),
        "sloka_count": sum(ch["sloka_count"] for ch in adhyayas),
        "adhyayas": adhyayas,
    })

corpus = {
    "work": {
        "title":            "Shrimad Bhagavata Purana",
        "title_devanagari": to_devanagari("shrImad bhAgavata purANa"),
        "title_iast":       to_iast("shrImad bhAgavata purANa"),
        "invocation":       invocation,
        "colophon":         colophon,
        "generated_utc":    dt.datetime.now(dt.timezone.utc).isoformat(timespec="seconds"),
    },
    "structure": {
        "levels": ["skandha", "adhyaya", "sloka", "pada"],
        "id_format": "<skandha>.<adhyaya>.<sloka>",
        "skandha_count": len(skandhas),
        "adhyaya_count": sum(sk["adhyaya_count"] for sk in skandhas),
        "sloka_count":   sum(sk["sloka_count"] for sk in skandhas),
        "vachana_count": n_vachana,
        "record_count":  len(slokas),
        "variant_count": sum(len(r.get("variants", [])) for r in slokas),
        "flagged_count": sum(1 for r in slokas if r.get("anomalies")),
        "notes": [
            "`text` holds verse text only. Speaker attributions and prose lead-ins are "
            "in `vachana` (typed `speaker` or `lead_in`) on the sloka they introduce.",
            "`text.padas` preserves the source's pāda numbering; (1, 3) is a standard anuṣṭubh.",
            "Records with `type: vachana_only` carry a dangling lead-in and `text: null`.",
            "`devanagari`/`iast` keep the source line breaks; `*_flat` are single-line joins.",
            "`skandhas[].adhyayas[].heading` and `.colophon` hold the structural "
            "vachanas (`atha prathamo'dhyayah`, `iti ... adhyayah`) taken from the "
            "per-skandha edition, which the combined file does not carry. `null` where "
            "that edition omits them: 4.2, 7.13 and 10.25.",
            "`skandhas[0].mangala` is a reciter's preliminary prayer printed before "
            "skandha 1, not Purana text. Two of its verses repeat 12.13.22 and 12.13.23.",
            "The `mahatmya` section is separate from `skandhas` so the canonical 12 / 335 "
            "counts stay exact. Its ids are `M.<adhyaya>.<sloka>`.",
            "Verse text comes from the combined edition; only apparatus comes from the "
            "per-skandha edition, because the two disagree slightly on verse splitting "
            "and rebuilding verses would shift the translation and chandas alignment.",
            "`variants` holds readings the source printed under the same verse number but "
            "attributed to another edition (marked `##vedabase ...##`); they are kept out "
            "of the verse body so two different verses are never spliced together.",
            "`anomalies` flags source irregularities (a pāda number used twice, a variant "
            "reading present) instead of silently normalising them.",
            "Translation `grouped: true` means the source rendered a verse range as one "
            "block; the same text is attached to each verse of that range, and `ref` "
            "records the range.",
        ],
    },
    "sources": {
        "sanskrit_apparatus": {
            "name": "sanskritdocuments.org per-skandha ITRANS edition",
            "url": PS_URL.format(stem="NN"),
            "used_for": ("adhyaya headings and colophons, the skandha 1 mangala block, "
                         "and the whole Mahatmya"),
            "note": "More recently proofread than the combined file, and not verse-aligned to it.",
        },
        "sanskrit": {
            "name": "sanskritdocuments.org ITRANS e-text",
            "url": ITX_URL,
            "script": "Devanagari (converted from ITRANS), with IAST and ITRANS retained",
            "rights": ("Prepared by volunteers for personal study and research; not to be "
                       "reposted commercially."),
        },
        "translations": SOURCE_META,
    },
    "mahatmya": {
        "title": to_devanagari("shrImadbhAgavatamAhAtmyam"),
        "title_iast": to_iast("shrImadbhAgavatamAhAtmyam"),
        "note": ("The Bhagavata Mahatmya. Its own colophon places it in the Padma "
                 "Purana, Uttarakhanda, so it frames the Bhagavata rather than "
                 "belonging to it. Kept separate from the 12 skandhas for that "
                 "reason, and absent from the combined bhagpur.itx entirely."),
        "source": PS_URL.format(stem="00-mahatmyam"),
        "invocation": render_ps(mh_inv) if mh_inv else None,
        "mangala": [render_ps(m) for m in mh_mangala] or None,
        "adhyaya_count": len(mahatmya_adhyayas),
        "sloka_count": n_mh,
        "translations": {"tagare": {"slokas": n_mh_tr,
                                    "fraction": round(n_mh_tr / n_mh, 4)}},
        "adhyayas": mahatmya_adhyayas,
    },
    "coverage": {
        "tagare":       {"slokas": n_t, "fraction": round(n_t / target, 4)},
        "anand_aadhar": {"slokas": n_a, "fraction": round(n_a / target, 4)},
        "both":         sum(1 for r in slokas if len(r["translations"]) == 2),
        "neither":      sum(1 for r in slokas
                            if r["type"] == "sloka" and not r["translations"]),
    },
    "skandhas": skandhas,
}

print(json.dumps({k: v for k, v in corpus["structure"].items() if k != "notes"}, indent=2))

{
  "levels": [
    "skandha",
    "adhyaya",
    "sloka",
    "pada"
  ],
  "id_format": "<skandha>.<adhyaya>.<sloka>",
  "skandha_count": 12,
  "adhyaya_count": 335,
  "sloka_count": 14088,
  "vachana_count": 1332,
  "record_count": 14089,
  "variant_count": 8,
  "flagged_count": 18
}


---
## 12. Quality checks

Fail loudly rather than shipping a corpus with silent holes.

In [25]:
problems = []

# The checks below iterate `slokas`, the Purana record list. A later cell that
# reused that name would quietly shrink every check, so pin it down first.
assert len(slokas) == len(by_id) == 14089, f"`slokas` is not the corpus: {len(slokas)}"
assert all(not r["id"].startswith("M.") for r in slokas), "Mahatmya leaked into `slokas`"

# a) structure
if corpus["structure"]["adhyaya_count"] != N_ADHYAYAS:
    problems.append(f"adhyaya count {corpus['structure']['adhyaya_count']} != {N_ADHYAYAS}")

# b) every sloka has non-empty Devanagari, and it really is Devanagari
DEVA = re.compile(r"[\u0900-\u097F]")
bad_script = [r["id"] for r in slokas
              if r["type"] == "sloka" and not DEVA.search(r["text"]["devanagari"])]
if bad_script:
    problems.append(f"{len(bad_script)} slokas without Devanagari: {bad_script[:5]}")

# c) no residual LaTeX / ITRANS control junk leaked into the rendered text
JUNK = re.compile(r"\\[a-zA-Z]+|##|\{|\}")
junk = [r["id"] for r in slokas
        if r["type"] == "sloka" and JUNK.search(r["text"]["devanagari"])]
junk += [f"{r['id']}~variant" for r in slokas
         for var in r.get("variants", []) if JUNK.search(var["devanagari"])]
if junk:
    problems.append(f"{len(junk)} slokas contain markup residue: {junk[:5]}")

# h) apparatus and Mahatmya
# The per-skandha edition omits exactly three adhyaya colophons. Assert the set,
# not a threshold, so a new omission or a parser regression both surface.
COLOPHON_GAPS = {"4.2", "7.13", "10.25"}
app_missing = {f"{sk['skandha']}.{ad['adhyaya']}"
               for sk in corpus["skandhas"] for ad in sk["adhyayas"]
               if not ad["colophon"]}
app_have = N_ADHYAYAS - len(app_missing)
print(f"adhyayas carrying a colophon: {app_have} / {N_ADHYAYAS}")
print(f"omitted by the source       : {sorted(app_missing)}")
if app_missing != COLOPHON_GAPS:
    problems.append(f"colophon gaps changed: {sorted(app_missing)} "
                    f"vs expected {sorted(COLOPHON_GAPS)}")
app_headings = sum(1 for sk in corpus["skandhas"] for ad in sk["adhyayas"] if ad["heading"])
print(f"adhyayas carrying a heading  : {app_headings} / {N_ADHYAYAS}")

mh = corpus["mahatmya"]
print(f"Mahatmya: {mh['adhyaya_count']} adhyayas, {mh['sloka_count']} slokas, "
      f"Tagare {mh['translations']['tagare']['slokas']}")
if mh["adhyaya_count"] != 6 or mh["sloka_count"] != 504:
    problems.append(f"Mahatmya shape unexpected: {mh['adhyaya_count']}/{mh['sloka_count']}")
mh_bad = [r["id"] for ad in mh["adhyayas"] for r in ad["slokas"]
          if not DEVA.search(r["text"]["devanagari"])]
if mh_bad:
    problems.append(f"{len(mh_bad)} Mahatmya slokas without Devanagari: {mh_bad[:5]}")
if corpus["structure"]["adhyaya_count"] != N_ADHYAYAS:
    problems.append("adding the Mahatmya changed the canonical adhyaya count")

flagged = [r["id"] for r in slokas if r.get("anomalies")]
print(f"records flagged with source anomalies: {len(flagged)} -> {flagged}")

# d) no vachana text left inside a sloka body
UVACA = re.compile(r"उवाच|ऊचुः")
leaked = [r["id"] for r in slokas
          if r["type"] == "sloka" and UVACA.search(r["text"]["devanagari"])]
print(f"slokas whose verse text contains उवाच/ऊचुः: {len(leaked)} {leaked[:5]}")
print("  (a few are genuine in-verse occurrences, not stray headers)")

# e) the closing || OM tatsat || must NOT have been glued onto the last verse
last = by_id["12.13.23"]["text"]["devanagari"]
if "तत्सत्" in last:
    problems.append("colophon leaked into 12.13.23")
print(f"\n12.13.23 last pāda: {last.splitlines()[-1]}")
print(f"colophon captured separately: {corpus['work']['colophon']['devanagari'] if corpus['work']['colophon'] else None}")

# f) translation text sanity: no boilerplate leakage
BOILER = re.compile(r"Creative Commons|Filognostic|wisdomlib|Vedabase|Buy now", re.I)
bleed = [(r["id"], k) for r in slokas for k, t in r["translations"].items()
         if BOILER.search(t["text"])]
if bleed:
    problems.append(f"{len(bleed)} translations contain site boilerplate: {bleed[:5]}")

# g) no residual double-encoded text survived the repair pass
moji = [(r["id"], k) for r in slokas for k, t in r["translations"].items()
        if _MOJI_HINT.search(t["text"])]
if moji:
    problems.append(f"{len(moji)} translations still double-encoded: {moji[:5]}")
print(f"translations with residual mojibake: {len(moji)}")

short = [(r["id"], k) for r in slokas for k, t in r["translations"].items()
         if len(t["text"]) < 15]
print(f"suspiciously short translations (<15 chars): {len(short)} {short[:5]}")

print("\n" + ("PROBLEMS:\n  " + "\n  ".join(problems) if problems else "All hard checks passed."))
assert not problems, problems

adhyayas carrying a colophon: 332 / 335
omitted by the source       : ['10.25', '4.2', '7.13']
adhyayas carrying a heading  : 327 / 335
Mahatmya: 6 adhyayas, 504 slokas, Tagare 498
records flagged with source anomalies: 18 -> ['1.7.1', '3.4.3', '3.14.49', '3.32.21', '4.29.46', '4.29.47', '4.29.76', '4.29.77', '5.17.17', '5.26.2', '6.6.22', '8.12.47', '8.16.23', '10.62.2', '10.65.25', '10.81.32', '11.27.41', '12.1.27']
slokas whose verse text contains उवाच/ऊचुः: 48 ['1.7.43', '1.19.19', '2.7.19', '3.15.31', '3.23.50']
  (a few are genuine in-verse occurrences, not stray headers)

12.13.23 last pāda: प्रणामो दुःखशमनस्तं नमामि हरिं परम्
colophon captured separately: ॥ ॐ तत्सत् ॥


translations with residual mojibake: 0
suspiciously short translations (<15 chars): 0 []

All hard checks passed.


In [26]:
# Per-skandha summary table
print(f"{'skandha':>8} {'adhyayas':>9} {'slokas':>8} {'vachanas':>9} {'Tagare':>9} {'Aadhar':>9}")
for sk in corpus["skandhas"]:
    recs = [r for ch in sk["adhyayas"] for r in ch["slokas"]]
    real = [r for r in recs if r["type"] == "sloka"]
    t = sum(1 for r in real if "tagare" in r["translations"])
    a = sum(1 for r in real if "anand_aadhar" in r["translations"])
    vv = sum(1 for r in recs if r["vachana"])
    print(f"{sk['skandha']:>8} {sk['adhyaya_count']:>9} {len(real):>8} {vv:>9} "
          f"{t:>9} {a:>9}")

tot = [r for r in slokas if r["type"] == "sloka"]
print(f"{'TOTAL':>8} {corpus['structure']['adhyaya_count']:>9} {len(tot):>8} "
      f"{n_vachana:>9} {n_t:>9} {n_a:>9}")

 skandha  adhyayas   slokas  vachanas    Tagare    Aadhar
       1        19      813        74       807       808
       2        10      391        24       387       391
       3        33     1413       140      1400      1411
       4        31     1445       168      1435      1444
       5        26      660        58       659       660
       6        19      851       105       845       851
       7        15      750        86       742       749
       8        24      933       101       913       930
       9        24      965        60       951       960
      10        90     3936       358      3907      3932
      11        31     1366       118      1341      1366
      12        13      565        40       557       564
   TOTAL       335    14088      1332     13945     14067


---
## 13. Write the deliverables

In [27]:
with OUT_JSON.open("w", encoding="utf-8") as fh:
    json.dump(corpus, fh, ensure_ascii=False, indent=1)

def jsonl_rows(corpus):
    for sk in corpus["skandhas"]:
        for ch in sk["adhyayas"]:
            for r in ch["slokas"]:
                yield sk["name"], ch["titles"]["tagare"] or ch["titles"]["anand_aadhar"], r
    mh = corpus["mahatmya"]
    for ch in mh["adhyayas"]:
        for r in ch["slokas"]:
            yield "Mahatmya (Padma Purana, Uttarakhanda)", ch["titles"]["tagare"], r


with OUT_JSONL.open("w", encoding="utf-8") as fh:
    for sk_name, ad_title, r in jsonl_rows(corpus):
        fh.write(json.dumps({
            "id": r["id"],
            "part": r.get("part", "purana"),
            "skandha": r["skandha"],
            "adhyaya": r["adhyaya"],
            "sloka": r["sloka"],
            "type": r["type"],
            "skandha_name": sk_name,
            "adhyaya_title": ad_title,
            "vachana_devanagari": r["vachana"]["devanagari"] if r["vachana"] else None,
            "vachana_type": r["vachana"]["type"] if r["vachana"] else None,
            "devanagari": r["text"]["devanagari_flat"] if r["text"] else None,
            "iast": r["text"]["iast_flat"] if r["text"] else None,
            "translation_tagare": (r["translations"].get("tagare") or {}).get("text"),
            "translation_anand_aadhar": (r["translations"].get("anand_aadhar") or {}).get("text"),
        }, ensure_ascii=False) + "\n")

for p in (OUT_JSON, OUT_JSONL):
    print(f"{p.relative_to(ROOT)}  {p.stat().st_size / 1e6:.1f} MB")

data/bhagavata_purana.json  42.0 MB
data/bhagavata_purana_flat.jsonl  18.1 MB


In [28]:
# Final read-back: reload from disk and show a complete record.
reloaded = json.loads(OUT_JSON.read_text(encoding="utf-8"))
print("reloaded:", reloaded["structure"]["sloka_count"], "slokas,",
      reloaded["structure"]["adhyaya_count"], "adhyayas\n")

sample = next(r for sk in reloaded["skandhas"] if sk["skandha"] == 1
              for ch in sk["adhyayas"] if ch["adhyaya"] == 1
              for r in ch["slokas"] if r["id"] == "1.1.6")
print(json.dumps(sample, ensure_ascii=False, indent=2)[:2400])

reloaded: 14088 slokas, 335 adhyayas

{
  "id": "1.1.6",
  "skandha": 1,
  "adhyaya": 1,
  "sloka": 6,
  "type": "sloka",
  "vachana": {
    "type": "speaker",
    "devanagari": "ऋषय ऊचुः ।",
    "iast": "ṛṣaya ūcuḥ |",
    "itrans": "R^iShaya UchuH |"
  },
  "text": {
    "devanagari": "त्वया खलु पुराणानि सेतिहासानि चानघ\nआख्यातान्यप्यधीतानि धर्मशास्त्राणि यान्युत",
    "iast": "tvayā khalu purāṇāni setihāsāni cānagha\nākhyātānyapyadhītāni dharmaśāstrāṇi yānyuta",
    "itrans": "tvayA khalu purANAni setihAsAni chAnagha\nAkhyAtAnyapyadhItAni dharmashAstrANi yAnyuta",
    "devanagari_flat": "त्वया खलु पुराणानि सेतिहासानि चानघ आख्यातान्यप्यधीतानि धर्मशास्त्राणि यान्युत",
    "iast_flat": "tvayā khalu purāṇāni setihāsāni cānagha ākhyātānyapyadhītāni dharmaśāstrāṇi yānyuta",
    "padas": [
      {
        "pada": 1,
        "devanagari": "त्वया खलु पुराणानि सेतिहासानि चानघ",
        "iast": "tvayā khalu purāṇāni setihāsāni cānagha",
        "itrans": "tvayA khalu purANAni setihAsAni chAnag

In [29]:
# Human-readable spot check across the whole work.
for vid in ("1.1.1", "2.9.33", "7.5.23", "10.14.8", "11.29.34", "12.13.23"):
    r = by_id[vid]
    print("=" * 78)
    print(f"{vid}")
    if r["vachana"]:
        print(f"  [{r['vachana']['type']}] {r['vachana']['devanagari']}")
    print("  " + r["text"]["devanagari"].replace("\n", "\n  "))
    print(f"  IAST: {r['text']['iast_flat'][:150]}")
    for k in ("tagare", "anand_aadhar"):
        t = r["translations"].get(k)
        if t:
            print(f"  [{k}] {t['text'][:260]}{'…' if len(t['text']) > 260 else ''}")
    print()

1.1.1
  जन्माद्यस्य यतोऽन्वयादितरतश्चार्थेष्वभिज्ञः स्वराट्
  तेने ब्रह्म हृदा य आदिकवये मुह्यन्ति यत्सूरयः
  तेजोवारिमृदां यथा विनिमयो यत्र त्रिसर्गोऽमृषा
  धाम्ना स्वेन सदा निरस्तकुहकं सत्यं परं धीमहि
  IAST: janmādyasya yato'nvayāditarataścārtheṣvabhijñaḥ svarāṭ tene brahma hṛdā ya ādikavaye muhyanti yatsūrayaḥ tejovārimṛdāṃ yathā vinimayo yatra trisargo'm
  [tagare] Let us meditate upon the Supreme Spirit who is real; from whom emanate the creation etc. (i.e. creation, preservation and destruction) of this (universe), (as can be inferred from) his presence in all that exists and his absence from all that is non-existent; …
  [anand_aadhar] Let there be the salutation of the original appearance of Him, Vâsudeva, the Fortunate One, from whom, being present here and in the beyond, for the purpose of recollection and full independence, the Vedic knowledge was imparted in the heart of the first creat…

2.9.33
  ऋतेऽर्थं यत्प्रतीयेत न प्रतीयेत चात्मनि
  तद्विद्यादात्मनो मायां यथाभासो यथा

---
## 14. Schema reference

The four hierarchy levels are named `skandha`, `adhyaya`, `sloka` and `pada` throughout.
Because `sloka` is the verse *number*, the verse *text* lives under `text`.

```
corpus
├── work            title, Devanāgarī/IAST titles, invocation, colophon, timestamp
├── structure       level names, ID format, counts, parsing notes
├── sources         Sanskrit e-text + both translations, with URLs and rights
├── coverage        translation coverage counts and fractions
└── skandhas[12]
    ├── skandha, name, adhyaya_count, sloka_count
    └── adhyayas[n]
        ├── skandha, adhyaya, titles{tagare, anand_aadhar}, sloka_count
        └── slokas[n]
            ├── id                "10.14.8"
            ├── skandha           10        (int)
            ├── adhyaya           14        (int)
            ├── sloka             8         (int)
            ├── type              "sloka" | "vachana_only"
            ├── vachana           null | {type: "speaker"|"lead_in", devanagari, iast, itrans}
            ├── text              null | {devanagari, iast, itrans,
            │                             devanagari_flat, iast_flat,
            │                             padas[{pada, devanagari, iast, itrans}]}
            ├── variants          (optional) other-edition readings under the same number
            ├── anomalies         (optional) notes on source irregularities
            └── translations
                ├── tagare        {text, ref, grouped, source}
                └── anand_aadhar  {text, ref, grouped, source}
```

`data/bhagavata_purana_flat.jsonl` is the same content flattened to one sloka per line,
keyed `skandha` / `adhyaya` / `sloka`, with the Devanāgarī, IAST and both translations as
top-level string fields, convenient for `pandas.read_json(..., lines=True)` or a
HuggingFace `datasets` load.

### Re-running

All network responses are cached under `data/raw/`. Re-executing the notebook rebuilds the
JSON from cache in seconds and makes no requests. Delete a cached file (or pass
`force=True` to `fetch`) to refresh a single page.